# 25 · 200 行手撸最小 Agent（ReAct 范式）

> **学习目标**：不依赖任何框架，200 行内手写一个**能用的** Agent，包含 ReAct loop + 2 个工具 + 最大步数 + token 预算。理解 Agent **不是框架**，本质就是「LLM + tools + loop」。
>
> **预备**：Foundations 章节走完（懂 asyncio + LLM 心智模型）。
>
> **为什么重要**：所有 LangChain Agents / LangGraph / CrewAI 都是这 200 行的包装。先手写一次，再看任何框架代码都不发懵。

**核心循环**（永远是这 5 步）：

```
1. 观察当前状态
2. LLM 思考下一步（Thought）
3. 决定动作（Action）→ 调工具 或 输出最终答案
4. 把工具结果加回上下文（Observation）
5. 判断是否完成；没完成回到 1
```

In [ ]:
MODE = 'OFFLINE'        # 'ONLINE' 用本机 Ollama（先 ollama serve）

import re, json, time, requests
from datetime import datetime
from typing import Callable
OLLAMA = 'http://127.0.0.1:11434'
print(f'MODE = {MODE}')

## 1. 定义工具 —— Agent 的「手」

**工具设计 4 原则**（02-Agent 章节核心）：
1. **小而专** —— 一个工具做一件事，名字清晰
2. **可观测** —— 每次调用都打 log
3. **可幂等** —— 能重试就不留副作用
4. **错误明确** —— 失败返回结构化错误而非异常

In [ ]:
def tool_calculate(expr: str) -> str:
    """计算一个 Python 数学表达式，仅允许数字与基本运算符。"""
    # 安全：白名单字符；杜绝 eval 注入
    if not re.fullmatch(r'[\d\s+\-*/().%]+', expr):
        return f'ERROR: 表达式含不允许字符: {expr!r}'
    try:
        return str(eval(expr))   # 上面已过滤，可信
    except Exception as e:
        return f'ERROR: {type(e).__name__}: {e}'

def tool_current_time(_: str = '') -> str:
    """返回当前时间字符串。参数被忽略。"""
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S')

TOOLS = {
    'calculate': {
        'fn': tool_calculate,
        'desc': '计算数学表达式。输入: 像 "2+2" 或 "100*1.05" 的 Python 表达式。',
    },
    'current_time': {
        'fn': tool_current_time,
        'desc': '获取当前时间（YYYY-MM-DD HH:MM:SS 格式）。输入: 空字符串。',
    },
}

# Smoke
for name, t in TOOLS.items():
    print(f'{name}: {t["desc"]}')
print('\ntest:', tool_calculate('2 * (3 + 4)'), '|', tool_current_time())

## 2. ReAct 提示词 —— Agent 的「脑」

**ReAct 格式**（[论文 2022](https://arxiv.org/abs/2210.03629) 提出）：让 LLM 把每一步思考 + 行动写在固定格式里。**好处**：你 grep 就能解析出工具调用。

```
Thought: 我需要先获取当前时间
Action: current_time
Action Input: 
Observation: 2026-06-04 14:23:00
Thought: 现在我能算出从今到 2030 还有几天
Action: calculate
Action Input: (2030 - 2026) * 365
Observation: 1460
Thought: 我已经有答案了
Final Answer: 大约 1460 天
```

In [ ]:
SYSTEM_PROMPT = '''你是一个严谨的 Agent。仅使用下面提供的工具回答问题，遵循 ReAct 格式：

可用工具:
{tools_desc}

你的每一步必须严格输出以下格式之一：

Thought: <你的思考>
Action: <工具名，必须是 {tool_names} 之一>
Action Input: <工具的输入>

—— 或者，当你能回答时：

Thought: <你的最终思考>
Final Answer: <给用户的答案>

不要输出 Observation —— 那是系统回填给你的。
'''

def build_system(tools: dict) -> str:
    tools_desc = '\n'.join(f'- {name}: {t["desc"]}' for name, t in tools.items())
    return SYSTEM_PROMPT.format(tools_desc=tools_desc, tool_names=list(tools))

## 3. LLM 抽象：OFFLINE 规则模拟 vs ONLINE Ollama

**OFFLINE 模拟规则**：扫上下文找最近一条用户 query，按关键词选工具，**模拟出 ReAct 格式响应**。

**ONLINE 真 LLM**：调 Ollama 的 OpenAI 兼容接口 `/v1/chat/completions`，让它输出 ReAct 格式。

In [ ]:
def llm_offline(messages: list[dict]) -> str:
    """按规则模拟 LLM 输出。看最近一条用户问题 + 所有 observation 来决定下一步。"""
    # 找 user 最近一条 query
    user_q = next((m['content'] for m in reversed(messages) if m['role'] == 'user'), '')
    # 已经看过几个 observation
    transcript = '\n'.join(m['content'] for m in messages if m['role'] == 'assistant')
    n_obs = transcript.count('Observation:')

    # 启发式：先看「需要时间」的关键词
    needs_time = re.search(r'(现在|今天|当前|now|today|time)', user_q)
    # 提取算式（最大化匹配）
    expr_match = re.search(r'([\d\s+\-*/().]{3,})', user_q)
    has_expr = expr_match and re.search(r'[+\-*/]', expr_match.group(1))

    if needs_time and n_obs == 0:
        return 'Thought: 用户问到时间，先获取当前时间\nAction: current_time\nAction Input: '
    if has_expr and ('Observation:' not in transcript or 'calculate' not in transcript):
        expr = expr_match.group(1).strip()
        return f'Thought: 这是数学题，调 calculate\nAction: calculate\nAction Input: {expr}'
    # 否则给出 Final Answer
    # 从 transcript 抽最近一条 Observation
    last_obs = re.findall(r'Observation:\s*(.*)', transcript)
    if last_obs:
        return f'Thought: 已得到工具结果\nFinal Answer: 结果是 {last_obs[-1]}'
    return f'Thought: 不需要工具就能回答\nFinal Answer: 关于 "{user_q}"，我无法在 OFFLINE 模式给详细回答。请切到 ONLINE 模式。'

def llm_online(messages: list[dict]) -> str:
    """调 Ollama 的 OpenAI 兼容接口。"""
    # Ollama: http://127.0.0.1:11434/v1/chat/completions
    r = requests.post(f'{OLLAMA}/v1/chat/completions', json={
        'model': 'qwen1.5_1.8',
        'messages': messages,
        'temperature': 0,
        'stop': ['Observation:'],   # 让 LLM 不要自己编 Observation
    }, timeout=120)
    r.raise_for_status()
    return r.json()['choices'][0]['message']['content']

if MODE == 'ONLINE':
    try:
        requests.get(f'{OLLAMA}/api/tags', timeout=1).raise_for_status()
        llm = llm_online
        print('✅ ONLINE：用 Ollama 真 LLM')
    except Exception:
        print('⚠ Ollama 未启动，降级 OFFLINE')
        MODE = 'OFFLINE'
if MODE == 'OFFLINE':
    llm = llm_offline
    print('使用 OFFLINE 规则模拟 LLM')

## 4. Agent loop —— 5 步循环的工程实现

**关键守则**：
- **最大步数**（max_iter）必须有，否则无限循环烧 token
- **token / 步数预算**：超就强制 Final Answer
- **解析失败要兜底**：LLM 不按格式输出时 retry 或终止
- **trace 必打**：每步都记到 trace 数组，方便事后回放

In [ ]:
def parse_action(text: str) -> dict | None:
    """从 LLM 输出里解析出 Action + Input。若没有则返回 None。"""
    action_m = re.search(r'Action:\s*(\S+)', text)
    input_m  = re.search(r'Action Input:\s*(.*?)(?:\n|$)', text, re.DOTALL)
    if action_m and input_m:
        return {'tool': action_m.group(1).strip(), 'input': input_m.group(1).strip()}
    return None

def parse_final_answer(text: str) -> str | None:
    m = re.search(r'Final Answer:\s*(.*)', text, re.DOTALL)
    return m.group(1).strip() if m else None

def run_agent(question: str, tools: dict, max_iter: int = 6, verbose: bool = True) -> dict:
    """返回 dict: {'answer', 'trace': [...], 'iter'}"""
    messages = [
        {'role': 'system', 'content': build_system(tools)},
        {'role': 'user',   'content': question},
    ]
    trace = []
    for step in range(1, max_iter + 1):
        # 1. LLM 推理
        llm_out = llm(messages)
        trace.append({'step': step, 'llm': llm_out})
        if verbose:
            print(f'\n── step {step} ──')
            print(f'LLM> {llm_out.strip()}')

        # 2. 检查 Final Answer
        ans = parse_final_answer(llm_out)
        if ans:
            return {'answer': ans, 'trace': trace, 'iter': step}

        # 3. 否则解析 Action
        action = parse_action(llm_out)
        if not action:
            return {'answer': '（解析失败）' + llm_out, 'trace': trace, 'iter': step}
        tool_name, tool_input = action['tool'], action['input']
        if tool_name not in tools:
            obs = f'ERROR: 未知工具 {tool_name!r}'
        else:
            obs = tools[tool_name]['fn'](tool_input)
        trace[-1]['action'] = action
        trace[-1]['observation'] = obs
        if verbose:
            print(f'TOOL[{tool_name}]> {obs}')

        # 4. 把 assistant 输出 + observation 加回上下文
        messages.append({'role': 'assistant', 'content': llm_out + f'\nObservation: {obs}'})

    # 超步数：强制 Final Answer
    return {'answer': f'（达到最大步数 {max_iter}，未得出答案）', 'trace': trace, 'iter': max_iter}

In [ ]:
# 跑 3 个测试 query
for q in ['100 * 1.05 ** 5 等于多少', '现在几点了', '7 + 8']:
    print(f'\n{"="*50}\n📝 Q: {q}')
    result = run_agent(q, TOOLS, max_iter=4, verbose=True)
    print(f'\n✅ 答案 ({result["iter"]} 步): {result["answer"]}')

## 5. 看 trace —— Agent 的「黑盒打开」

**生产 Agent debug 的第一件事**：看 trace。每步 LLM 输出 + 工具调用 + 结果都要可回放。

In [ ]:
result = run_agent('现在几点了', TOOLS, max_iter=4, verbose=False)
print('Trace（结构化）：')
print(json.dumps(result['trace'], ensure_ascii=False, indent=2))
print(f'\nFinal Answer: {result["answer"]}')
print(f'Steps used:   {result["iter"]}')

## 深入思考

1. **为什么用文本格式（ReAct）解析而不是 JSON / function-calling？**
   - ReAct 是「**模型无关**」—— 任何 LLM 都能输出文本。function-calling 需要模型本身支持。教学先理解 ReAct，工程上用 26 号 notebook 的 OpenAI/Anthropic Tool Use 更稳。
2. **`stop=['Observation:']` 这个参数为什么关键？**
   - 不加 stop，LLM 会**自己续写 Observation 假装是工具结果** —— 经典「幻觉工具调用」。stop 强制 LLM 在 Action 输出后停手，让我们去真调工具。
3. **`max_iter=6` 怎么定？**
   - 经验：3-10。任务越复杂越大。**生产里通常加 token 预算 + 步数双限**，先到先停。
4. **如果 LLM 输出 `Action: calculator`（拼错了）怎么办？**
   - 我们的代码会返回 `ERROR: 未知工具 'calculator'`。**好处**：LLM 看到 Observation 是 ERROR 就有机会下一步纠正（这就是 ReAct 的健壮性来源）。
5. **OFFLINE 规则模拟有教学价值吗？**
   - 有。它强迫你**把「Agent 决策逻辑」写明白** —— 你的规则覆盖了哪些 case，没覆盖什么。换成 LLM 后这些 case 就是 LLM 必须搞定的。

**改一改**：
- 把 `max_iter=1`，看「无法在 1 步内解决」的 query 怎么表现
- 加一个 `tool_search_wiki` mock 工具（输入关键词，输出固定字符串），跑「Python 是什么」

## 自检 ✅

- [ ] 不查代码，10 分钟内白板画出 Agent loop 5 步
- [ ] 解释「ReAct vs function-calling」的差别与各自适用场景
- [ ] 解释 `stop=['Observation:']` 的作用
- [ ] 解释「为什么 max_iter 必须有」+ 经验取值
- [ ] 给一个看上去无限循环的 Agent 日志，立刻指出第一个排查方向（解析失败 / stop 没设 / max_iter 太大）

## 下一步

→ [`26_tool_use_openai_style.ipynb`](26_tool_use_openai_style.ipynb)